## Bi-Encoder with BERT

In [ ]:
# Global hyperparams
DATASET_SIZE = 'small' 
BATCH_SIZE =  128

# Padding / truncating
MAX_TITLE_LENGTH = 1 # set in embeddings
MAX_IMPRESSIONS = 25
MAX_HISTORY_LENGTH = 25

# Multi-head self-attention
DROPOUT = 0.2
HEAD_NUM = 4
HEAD_DIM = 64
INTERMEDIATE_DIM = 128

# Regularisation
LAMBDA_DIVERSITY = 350
LAMBDA_NOVELTY = 6

## Load the Dataset

In [2]:
from recs import *
data = DataLoader(BATCH_SIZE, data_size=DATASET_SIZE)
embedding_layer = BERTEmbedding(DATASET_SIZE)
dataset = data.get_train_ds() 
val_dataset = data.get_val()
data.get_popularity()

Available device: GPU 



## Build the News Encoder

In [3]:
title_input = keras.Input(shape=(MAX_TITLE_LENGTH,), dtype="int32", name='news_encoder_input')
y = embedding_layer(title_input)
y = layers.MultiHeadAttention(HEAD_NUM, HEAD_DIM, dropout=DROPOUT)(y, y)
title_vector = AttentionPooling()(y) 
news_encoder = keras.Model(title_input, title_vector, name="news_encoder")

## Build the User Encoder

In [4]:
history_input = keras.Input(shape=(MAX_HISTORY_LENGTH, 1), 
                            dtype="int32", name='user_history_encoder')

y = layers.TimeDistributed(news_encoder)(history_input)
y = layers.MultiHeadAttention(HEAD_NUM, HEAD_DIM)(y, y)

z = layers.Dense(INTERMEDIATE_DIM, activation='relu')(y) 
z = layers.Dense(ops.shape(y)[-1])(z)                    
z = layers.Add()([y, z])

encoded_user = AttentionPooling()(z)

user_encoder = keras.Model(history_input, encoded_user, name="user_encoder")

## Build the Main Model

In [ ]:
# Build model
history_input = keras.Input(
    shape=(MAX_HISTORY_LENGTH, 1), 
    dtype="int32", name='user_history_input')

candidate_input = keras.Input(
    shape=(MAX_IMPRESSIONS, 1), 
    dtype="int32", name='candidate_input') 

encoded_user = user_encoder(history_input) 
encoded_candidates = layers.TimeDistributed(news_encoder)(candidate_input)
logits = layers.Dot(axes=-1)([encoded_candidates, encoded_user])

logits = DiversityRegularisation(
    lambda_diversity=LAMBDA_DIVERSITY)(logits, encoded_candidates) 
logits = NoveltyRegulariser(
    lambda_novelty=LAMBDA_NOVELTY, 
    item_popularity=data.item_popularity, 
    total_users=data.total_users) (logits, candidate_input)

preds = layers.Softmax()(logits)
model = keras.Model([history_input, candidate_input], preds)

# model.summary(line_length=100, show_trainable=True, expand_nested=True)

## Training the model

In [6]:
model.compile(optimizer=keras.optimizers.Adam(5e-4), 
               loss=RankLoss(), metrics=[MaskedAUC()])

In [ ]:
# Test in eager mode
loss_fn = RankLoss()
inputs, labels = list(dataset.take(1))[0]

with tf.GradientTape() as tape:
    predictions = model(inputs, training=True)        
    rank_loss = loss_fn(labels, predictions)
    regularisation = model.compute_loss(y=labels, y_pred=predictions)
    
    print(f"Weighted regularisation: {regularisation.numpy()} \
            \nRank loss: {rank_loss}")

In [8]:
# Train
history = model.fit(
    dataset,     
    validation_data=val_dataset,
    epochs=5, 
    verbose=1,
    callbacks=[
    keras.callbacks.EarlyStopping(monitor='val_masked_auc', mode='max')],
    steps_per_epoch=data.steps_per_epoch,
    validation_steps=data.validation_steps,
)

Epoch 1/5


I0000 00:00:1743616844.015249  344548 service.cc:146] XLA service 0x7b93c8016420 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743616844.015274  344548 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 SUPER, Compute Capability 8.9


    3/17443 ━━━━━━━━━━━━━━━━━━━━ 11:46 41ms/step - loss: 999.1389 - masked_auc: 0.5255    

I0000 00:00:1743616875.120281  344548 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


17443/17443 ━━━━━━━━━━━━━━━━━━━━ 826s 40ms/step - loss: 941.9147 - masked_auc: 0.7561 - val_loss: 970.1178 - val_masked_auc: 0.7283
Epoch 2/5
17443/17443 ━━━━━━━━━━━━━━━━━━━━ 640s 37ms/step - loss: 931.4501 - masked_auc: 0.7816 - val_loss: 967.9549 - val_masked_auc: 0.7312
Epoch 3/5
17443/17443 ━━━━━━━━━━━━━━━━━━━━ 642s 37ms/step - loss: 928.9870 - masked_auc: 0.7863 - val_loss: 968.8420 - val_masked_auc: 0.7277


## Test and log results

In [10]:
test_dataset = data.get_test()
y_pred = model.predict(test_dataset)

log_results(y_true=data.test_labels, 
            y_pred=y_pred, 
            imprs=data.test_imprs, 
            dataset_size=DATASET_SIZE,
            modelname='Bi-encoderwBERT', 
            history=history)

Saved model predictions here: ../.data/Bi-encoderwBERT.npy


,Bi-encoderwBERT
modelname,Bi-encoderwBERT
dataset_size,large
timestamp,2025-04-02 19:50
auc,0.7289
mean_mrr,0.308
ndcg@5,0.3354
ndcg@10,0.4005
mean_epc,1.2614
mean_intra_list_diversity,0.2444
mean_surprisal,7.9398
